# CNN KWS: запись и инференс

1. Выполните ячейку с загрузкой модели (нужен файл `artifacts/kws_cnn.pt` из `lab3.ipynb`).
2. **Запись** — отдельная ячейка: пишет ~1 с в переменные `y_live`, `sr_live`.
3. **Прослушивание** — следующая ячейка: плеер для проверки записи перед инференсом.
4. **Инференс** — предсказание по `y_live`.

Чтобы проверить без микрофона, в ячейке записи можно загрузить WAV: см. комментарий.

**Почему live ≠ val/test:** другой микрофон, комната, громкость; в `extract_logmel` нормализация **на каждом** фрагменте усиливает шум — тишина может стать «yes». В ноутбуке — пиковая нормализация и порог по энергии; если уровень **ниже порога**, класс **`silence` выставляется без CNN** (эвристика), иначе сеть могла бы выдать любой класс с шума. Пороги: `silence_peak` / `silence_rms` в `prepare_live_audio`.


In [61]:
from pathlib import Path

import librosa
import numpy as np
import torch
import torch.nn as nn

SR = 16000
N_MELS = 64
N_FFT = 512
HOP_LENGTH = 160
CHECKPOINT = Path("artifacts/kws_cnn.pt")


class SmallKwsCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)


def extract_logmel(audio, sr=SR):
    if sr != SR:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=SR)
        sr = SR
    if len(audio) < sr:
        audio = np.pad(audio, (0, sr - len(audio)))
    elif len(audio) > sr:
        audio = audio[:sr]
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS, power=2.0
    )
    logmel = librosa.power_to_db(mel, ref=np.max)
    logmel = (logmel - logmel.mean()) / (logmel.std() + 1e-6)
    return logmel.astype(np.float32)


def load_checkpoint(path, device):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    labels = list(ckpt["labels"])
    model = SmallKwsCNN(num_classes=len(labels)).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    return model, labels


def prepare_live_audio(y, silence_peak=0.02, silence_rms=0.006):
    """Пиковая нормализация под датасет + отсечка почти цифровой тишины.

    Модель училась на клипах с нормальной громкостью; запись с микрофона часто тише.
    Если сигнал совсем слабый, не гоняем его через сеть (иначе per-utterance norm
    раздувает шум в «левый» класс).
    """
    y = np.asarray(y, dtype=np.float32).reshape(-1)
    peak = float(np.max(np.abs(y)))
    rms = float(np.sqrt(np.mean(np.square(y))))
    if peak < silence_peak and rms < silence_rms:
        return None, (
            f"Сигнал слишком слабый (peak={peak:.4f}, rms={rms:.4f}). "
            "Поднимите уровень микрофона или говорите ближе — иначе сеть «угадывает» с шума."
        )
    y = (y / (peak + 1e-8)) * 0.95
    return y, None


def predict_kws(model, labels, audio, sr, device, min_confidence=0.4):
    y_adapt, prep_msg = prepare_live_audio(audio)
    if y_adapt is None:
        # Тишина / слишком слабо: CNN на таком фрагменте даёт случайный класс.
        if "silence" in labels:
            ranked = {lbl: (1.0 if lbl == "silence" else 0.0) for lbl in labels}
            note = (
                "(Уровень ниже порога — класс «silence» назначен эвристикой, без нейросети.)\n"
                + prep_msg
            )
            return "silence", ranked, note
        return None, {}, prep_msg
    logmel = extract_logmel(y_adapt, sr=sr)
    x = torch.from_numpy(logmel).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    order = np.argsort(-probs)
    ranked = {labels[i]: float(probs[i]) for i in order}
    best = labels[int(order[0])]
    top_p = float(probs[int(order[0])])
    note = ""
    if top_p < min_confidence:
        note = (
            f"\n⚠ Низкая уверенность ({top_p:.2f}). "
            "Речь с микрофона сильно отличается от обучающих записей — попробуйте громче/чётче «stop»."
        )
    return best, ranked, note


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if not CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"Нет {CHECKPOINT}. Сохраните модель в lab3.ipynb (ячейка сохранения CNN)."
    )

cnn_live, labels_live = load_checkpoint(CHECKPOINT, device)
print("Классы:", labels_live)


Device: cuda
Классы: ['eight', 'five', 'four', 'nine', 'one', 'seven', 'silence', 'six', 'three', 'two', 'unknown']


## Запись голоса (~1 с)

Запустите ячейку и говорите команду во время записи. Результат: `y_live` (сигнал), `sr_live = 16000`.

*Альтернатива:* раскомментируйте загрузку WAV вместо микрофона.

In [171]:
import sounddevice as sd

RECORD_DURATION = 1.0  # секунды
sr_live = SR
n_samples = int(RECORD_DURATION * sr_live)

print(f"Запись {RECORD_DURATION:.1f} с @ {sr_live} Hz — говорите после старта.")
buf = sd.rec(n_samples, samplerate=sr_live, channels=1, dtype=np.float32)
sd.wait()
y_live = buf.flatten()

print("Готово, длина сэмплов:", len(y_live))

from IPython.display import Audio, display

display(Audio(y_live, rate=int(sr_live)))

# Вместо микрофона можно подставить файл:
# y_live, sr_live = librosa.load("path/to.wav", sr=None, mono=True)

Запись 1.0 с @ 16000 Hz — говорите после старта.
Готово, длина сэмплов: 16000


## Инференс

Использует `y_live` и `sr_live` из ячейки **записи** и модель из первой ячейки.

In [172]:
best, ranked, note = predict_kws(cnn_live, labels_live, y_live, sr_live, device)

if best is None:
    print(note)
else:
    print("Предсказание:", best)
    if note:
        print(note)
    print("\nВероятности по классам:")
    for name, p in ranked.items():
        print(f"  {name:12s}  {p:.4f}")
    if note and "без нейросети" in note:
        print("\n(Чтобы распознавать слова, а не только тишину — говорите громче или усильте микрофон.)")



Предсказание: eight

⚠ Низкая уверенность (0.25). Речь с микрофона сильно отличается от обучающих записей — попробуйте громче/чётче «stop».

Вероятности по классам:
  eight         0.2493
  unknown       0.2292
  three         0.2238
  two           0.1168
  nine          0.0475
  four          0.0427
  seven         0.0368
  six           0.0222
  one           0.0188
  five          0.0117
  silence       0.0014
